In [1]:
import pandas as pd
import numpy as np
import  requests 
from bs4 import BeautifulSoup
import time

In [2]:
df = pd.read_csv('data_movies_cleaned.csv')

In [3]:
#We are converting the 'Title' column from our DataFrame, which loaded the original dataset, into a Python list of movie titles. 
#This will enable us to loop through each film and perform web scraping on them individually.

movie_titles = df['Title'].tolist()

In [4]:
#creating an empty list called 'movie_urls' and we loop through each movie title

movie_urls = []
for title in movie_titles:
    formatted_title = title.replace(' ', '+')
    url = f"https://www.imdb.com/search/title/?title={formatted_title}"
    movie_urls.append(url)

In [5]:
movie_urls[:5]

['https://www.imdb.com/search/title/?title=Obsession',
 'https://www.imdb.com/search/title/?title=Mortal+Kombat+II',
 'https://www.imdb.com/search/title/?title=Michael',
 'https://www.imdb.com/search/title/?title=Karuppu',
 'https://www.imdb.com/search/title/?title=The+Unknown+Man']

In [6]:
#prepare to scrape the content from the first IMDb search URL

url_to_scrape = movie_urls[0]

headers = {'User-Agent': 'Mozilla/5.0'}

response = requests.get(url_to_scrape, headers=headers)
response.encoding = 'utf-8'
soup = BeautifulSoup(response.text, 'html.parser')

print(response.status_code)

202


In [7]:
print(response.url)

https://www.imdb.com/search/title/?title=Obsession


In [8]:
print(soup.prettify()[:500])

In [9]:
print(response.content)

b''


IMDb scraping was unsuccessful due to access restrictions, so we pivoted to Rotten Tomatoes as our web scraping source.

In [10]:
rt_url = "https://www.rottentomatoes.com/m/obsession_2023"

headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(rt_url, headers=headers)

print(response.status_code)

200


In [11]:
print(response.text[:500])

<!DOCTYPE html>
<html lang="en" dir="ltr" xmlns="http://www.w3.org/1999/xhtml" prefix="fb: http://www.facebook.com/2008/fbml og: http://opengraphprotocol.org/schema/">
    <head prefix="og: http://ogp.me/ns# flixstertomatoes: http://ogp.me/ns/apps/flixstertomatoes#">

        
            <script>
                const encodedData = sessionStorage.getItem('oneTrustSyncData');
                let id, token;

                if (encodedData) {
                    try {
                        cons


In [12]:
movie_data = {}

movie_data["rt_url"] = rt_url
movie_data["page_title"] = soup.title.text if soup.title else None

# Get all page text
page_text = soup.get_text(" ", strip=True)

movie_data["contains_audience_score"] = "audiencescore" in response.text.lower()
movie_data["contains_critic"] = "critic" in response.text.lower()
movie_data["contains_rating"] = "rating" in response.text.lower()
movie_data["contains_score"] = "score" in response.text.lower()

movie_data

{'rt_url': 'https://www.rottentomatoes.com/m/obsession_2023',
 'page_title': None,
 'contains_audience_score': True,
 'contains_critic': True,
 'contains_rating': True,
 'contains_score': True}

In [13]:
#turn into dataframe

scraped_df = pd.DataFrame([movie_data])

scraped_df

,rt_url,page_title,contains_audience_score,contains_critic,contains_rating,contains_score
0,https://www.rottentomatoes.com/m/obsession_2023,None,True,True,True,True


In [14]:
#turn into CSV file

scraped_df.to_csv("rottentomatoes_scraped_test.csv", index=False)

In [15]:
import os

os.listdir()

['.ipynb_checkpoints',
 'Cleaning Box office.ipynb',
 'Data wrangling.cleaned.ipynb',
 'data_movies_cleaned.csv',
 'data_wrangling.csv',
 'enriched_movies.csv',
 'page_source.txt',
 'rottentomatoes_scraped_test.csv',
 'rottentomatoes_scraped_test_v2.csv',
 'Web_scraping.ipynb',
 'wikipedia_awards_test.csv']

In [16]:
test_df = pd.read_csv("rottentomatoes_scraped_test.csv")

test_df

,rt_url,page_title,contains_audience_score,contains_critic,contains_rating,contains_score
0,https://www.rottentomatoes.com/m/obsession_2023,NaN,True,True,True,True


In [17]:
# Try to get title from different places

title_tag = soup.find("title")
meta_title = soup.find("meta", property="og:title")

if title_tag:
    page_title = title_tag.text.strip()
elif meta_title:
    page_title = meta_title.get("content")
else:
    page_title = None

page_title

In [18]:
movie_data = {}

movie_data["rt_url"] = rt_url
movie_data["page_title"] = page_title
movie_data["status_code"] = response.status_code
movie_data["contains_audience_score"] = "audiencescore" in response.text.lower()
movie_data["contains_critic"] = "critic" in response.text.lower()
movie_data["contains_rating"] = "rating" in response.text.lower()
movie_data["contains_score"] = "score" in response.text.lower()

scraped_df = pd.DataFrame([movie_data])
scraped_df

,rt_url,page_title,status_code,contains_audience_score,contains_critic,contains_rating,contains_score
0,https://www.rottentomatoes.com/m/obsession_2023,None,200,True,True,True,True


In [19]:
scraped_df.to_csv("rottentomatoes_scraped_test_v2.csv", index=False)

In [20]:
with open("page_source.txt", "w", encoding="utf-8") as f:
    f.write(response.text)

Although we successfully connected to movie pages (HTTP status code 200), the review and score data were not easily accessible through standard scraping techniques. Much of the information appeared to be dynamically loaded or stored in a structure that was difficult to extract within the project timeframe.
After evaluating alternative sources, we decided to pivot to Wikipedia.

Wikipedia provides more accessible and structured information that can be scraped using requests and BeautifulSoup.
Final Decision

We chose to focus on scraping awards and nominations information from Wikipedia.




In [3]:
df.head()

,TMDb_ID,IMDb_ID,Title,Year,Genre,Primary Genre,TMDb Rating,TMDb Votes,Popularity,Revenue,Budget,Runtime,IMDb Rating,IMDb Votes,Box Office
0,1339713,tt37287335,Obsession,2026,"Horror, Thriller",Horror,7.917,841,581.7554,297445815.0,750000.0,108,8.2,50183.0,104757650.0
1,931285,tt17490712,Mortal Kombat II,2026,"Action, Fantasy, Adventure",Action,7.988,1449,519.4298,128857238.0,80000000.0,116,6.7,36627.0,77767462.0
2,936075,tt11378946,Michael,2026,"Music, Drama",Music,8.653,2635,503.0910,934009960.0,250000000.0,128,7.7,107813.0,340068767.0
3,1367220,tt33988385,Karuppu,2026,"Action, Fantasy",Action,7.059,17,384.3573,3345079.0,16000000.0,152,NaN,NaN,NaN
4,879945,tt16288638,The Unknown Man,2021,Drama,Drama,7.800,9,325.1904,NaN,NaN,23,7.0,53.0,NaN


In [4]:
df.columns

Index(['TMDb_ID', 'IMDb_ID', 'Title', 'Year', 'Genre', 'Primary Genre',
       'TMDb Rating', 'TMDb Votes', 'Popularity', 'Revenue', 'Budget',
       'Runtime', 'IMDb Rating', 'IMDb Votes', 'Box Office'],
      dtype='str')

In [5]:
title = df.loc[0, "Title"]
year = df.loc[0, "Year"]

print(title, year)

Obsession 2026


In [6]:
headers = {"User-Agent": "Mozilla/5.0"}

search_url = "https://en.wikipedia.org/w/index.php"
params = {"search": f"{title} {year} film awards"}

response = requests.get(search_url, params=params, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

print(response.status_code)
print(soup.title.text)
print(response.url)

200
Obsession 2026 film awards - Search results - Wikipedia
https://en.wikipedia.org/w/index.php?search=Obsession+2026+film+awards


In [7]:
#Initial awards detection test
#This exploratory test searched for award-related keywords in the retrieved Wikipedia page.


page_text = soup.get_text(" ", strip=True).lower()

award_keywords = ["award", "awards", "nomination", "nominated", "won", "winner"]

has_awards = any(word in page_text for word in award_keywords)

print(has_awards)

True


In [8]:
print(response.status_code)
print(soup.title.text)
print(response.url)

200
Obsession 2026 film awards - Search results - Wikipedia
https://en.wikipedia.org/w/index.php?search=Obsession+2026+film+awards


In [9]:
#Theinvestigation showed that the page returned was a Wikipedia search results page rather than the movie page itself.
#so this approach was not retained in the final workflow.

In [10]:
df[["Title", "Year"]].head()

,Title,Year
0,Obsession,2026
1,Mortal Kombat II,2026
2,Michael,2026
3,Karuppu,2026
4,The Unknown Man,2021


In [11]:
df.shape

(815, 15)

In [12]:
df[["Title", "Year"]].head(10)

,Title,Year
0,Obsession,2026
1,Mortal Kombat II,2026
2,Michael,2026
3,Karuppu,2026
4,The Unknown Man,2021
5,Peddi,2026
6,Bhooth Bangla,2026
7,Hai Jawani Toh Ishq Hona Hai,2026
8,Disclosure Day,2026
9,Your Heart Will Be Broken,2026


In [13]:
#Awards information was collected for the Top 50 most popular movies in the dataset. 
#This approach focuses the analysis on the movies with the highest audience engagement and the most complete available information.

In [14]:
top50 = df.nlargest(50, "Popularity")

top50[["Title", "Year", "Popularity"]].head()

,Title,Year,Popularity
0,Obsession,2026,581.7554
1,Mortal Kombat II,2026,519.4298
2,Michael,2026,503.0910
3,Karuppu,2026,384.3573
6,Bhooth Bangla,2026,362.8242


In [15]:
#Some highly popular movies did not have a usable Wikipedia page, especially upcoming releases. 
#For this reason, we kept the Top 50 by Popularity as our target sample, then collected awards information only when a reliable Wikipedia page was available.

In [16]:
top50 = df.nlargest(50, "Popularity").copy()

top50[["Title", "Year", "Popularity"]].head(10)

,Title,Year,Popularity
0,Obsession,2026,581.7554
1,Mortal Kombat II,2026,519.4298
2,Michael,2026,503.0910
3,Karuppu,2026,384.3573
6,Bhooth Bangla,2026,362.8242
11,Salitan,2024,336.4305
4,The Unknown Man,2021,325.1904
5,Peddi,2026,324.8609
8,Disclosure Day,2026,286.9304
9,Your Heart Will Be Broken,2026,276.5688


In [17]:
#Wikipedia Page Availability

#For the Top 50 most popular movies, we first checked whether a usable Wikipedia page was available. 
#When no reliable Wikipedia page was found, the awards information was left as missing (`NaN`) instead of forcing uncertain data from another source.

In [18]:
headers = {"User-Agent": "Mozilla/5.0"}

def check_wikipedia_page(title, year):
    search_url = "https://en.wikipedia.org/w/index.php"
    
    params = {
        "search": f"{title} {year} film"
    }
    
    response = requests.get(search_url, params=params, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")
    
    page_title = soup.title.text if soup.title else np.nan
    final_url = response.url
    
    # If Wikipedia returns a search results page, we consider no reliable page found
    if "Search results" in page_title:
        return {
            "wikipedia_page_found": False,
            "wikipedia_url": np.nan,
            "wikipedia_title": page_title
        }
    
    return {
        "wikipedia_page_found": True,
        "wikipedia_url": final_url,
        "wikipedia_title": page_title
    }

In [19]:
test_results = []

for index, row in top50.head(5).iterrows():
    result = check_wikipedia_page(row["Title"], row["Year"])
    result["Title"] = row["Title"]
    result["Year"] = row["Year"]
    test_results.append(result)
    time.sleep(1)

test_df = pd.DataFrame(test_results)
test_df

,wikipedia_page_found,wikipedia_url,wikipedia_title,Title,Year
0,False,NaN,Obsession 2026 film - Search results - Wikipedia,Obsession,2026
1,False,NaN,Mortal Kombat II 2026 film - Search results - ...,Mortal Kombat II,2026
2,False,NaN,Michael 2026 film - Search results - Wikipedia,Michael,2026
3,False,NaN,Karuppu 2026 film - Search results - Wikipedia,Karuppu,2026
4,False,NaN,Bhooth Bangla 2026 film - Search results - Wik...,Bhooth Bangla,2026


#After exploring the most popular movies in the dataset, 
#we observed that many of them were upcoming releases with limited publicly available information. 
#To improve data availability and consistency, we focused our web scraping efforts on movies released before 2024.

In [20]:
released_movies = df[df["Year"] < 2024].copy()

released_movies.shape

(466, 15)

In [21]:
released_movies = released_movies.nlargest(50, "Popularity")

released_movies[["Title", "Year", "Popularity"]].head(20)

,Title,Year,Popularity
4,The Unknown Man,2021,325.1904
26,Graphic Desires,2023,137.9446
53,Scary Movie,2000,71.7195
54,Interstellar,2014,70.1924
56,The Devil Wears Prada,2006,63.4911
62,Real Steel,2011,61.5234
70,After We Fell,2021,53.1990
71,Addicted,2014,52.8071
79,Room in Rome,2010,52.2855
66,Avengers: Infinity War,2018,51.7509


In [22]:
released_movies.iloc[0]

TMDb_ID                   879945
IMDb_ID               tt16288638
Title            The Unknown Man
Year                        2021
Genre                      Drama
Primary Genre              Drama
TMDb Rating                  7.8
TMDb Votes                     9
Popularity              325.1904
Revenue                      NaN
Budget                       NaN
Runtime                       23
IMDb Rating                  7.0
IMDb Votes                  53.0
Box Office                   NaN
Name: 4, dtype: object

## Popularity Metric Observation

During the exploration phase, we observed that some movies with relatively low vote counts appeared among the most popular titles in the dataset.

example: The Unknown Man
IMDb Votes = 53
TMDb Votes = 9
Popularity = 325

This is likely due to the TMDb Popularity metric, which reflects recent activity and engagement on the platform rather than overall audience size or long-term popularity.

This observation was considered when evaluating different sampling strategies for the web scraping phase.

In [23]:
sample_movies = released_movies.nlargest(50, "IMDb Votes").copy()

sample_movies[["Title", "Year", "IMDb Votes"]].head(20)

,Title,Year,IMDb Votes
86,The Dark Knight,2008,3161907.0
145,Inception,2010,2811614.0
54,Interstellar,2014,2516752.0
116,The Lord of the Rings: The Fellowship of the Ring,2001,2203785.0
103,The Lord of the Rings: The Return of the King,2003,2161991.0
93,The Avengers,2012,1555308.0
122,Avatar,2009,1486308.0
125,Avengers: Endgame,2019,1442446.0
66,Avengers: Infinity War,2018,1368739.0
137,Parasite,2019,1148210.0


In [24]:
#The Dark Knight

headers = {"User-Agent": "Mozilla/5.0"}

url = "https://en.wikipedia.org/wiki/The_Dark_Knight"

response = requests.get(url, headers=headers)

print(response.status_code)
print(response.url)

200
https://en.wikipedia.org/wiki/The_Dark_Knight


In [25]:
soup = BeautifulSoup(response.text, "html.parser")

print(soup.title.text)

The Dark Knight - Wikipedia


In [26]:
page_text = soup.get_text(" ", strip=True)

award_keywords = [
    "Academy Award",
    "Academy Awards",
    "award",
    "awards",
    "nomination",
    "nominations",
    "won"
]

for keyword in award_keywords:
    print(keyword, ":", keyword.lower() in page_text.lower())

Academy Award : True
Academy Awards : True
award : True
awards : True
nomination : True
nominations : True
won : True


In [27]:
headings = soup.find_all(["h2", "h3"])

for heading in headings:
    print(heading.get_text(strip=True))

Contents
Plot
Cast
Production
Development
Writing
Casting
Pre-production
Filming in Chicago
Filming in England and Hong Kong
Post-production
Special effects and design
Music
Release
Marketing and anti-piracy
Context
Box office
Reception
Critical response
Accolades
Other releases
Home media
Merchandise and spin-offs
Themes and analysis
Terrorism and escalation
Morality and ethics
Legacy
Cultural influence
Retrospective assessments
Sequel
Notes
References
Citations
Works cited
External links


In [28]:
for heading in soup.find_all(["h2", "h3"]):
    title = heading.get_text(strip=True)

    if "Accolades" in title:
        print("FOUND:", title)
        
        next_element = heading.find_next()
        print(next_element.get_text(strip=True)[:1000])

FOUND: Accolades



In [29]:
for heading in soup.find_all(["h2", "h3"]):
    title = heading.get_text(strip=True)

    if "Accolades" in title:
        print(heading)

<h3 id="Accolades">Accolades</h3>


In [30]:
accolades = soup.find(id="Accolades")

print(accolades)

<h3 id="Accolades">Accolades</h3>


In [31]:
accolades_heading = soup.find("h3", id="Accolades")

accolades_text = []

for element in accolades_heading.find_next_siblings():
    if element.name in ["h2", "h3"]:
        break
    accolades_text.append(element.get_text(" ", strip=True))

accolades_text = " ".join(accolades_text)

print(accolades_text[:1500])

In [32]:
accolades_heading = soup.find("h3", id="Accolades")

print(accolades_heading.parent)

<div class="mw-heading mw-heading3"><h3 id="Accolades">Accolades</h3></div>


In [33]:
parent = accolades_heading.parent

for element in parent.find_next_siblings()[:10]:
    print("TAG:", element.name)
    print(element.get_text(" ", strip=True)[:500])
    print("---")

TAG: link

---
TAG: div
Main article: List of accolades received by The Dark Knight
---
TAG: figure
Heath Ledger (2006) received the Academy Award for Best Supporting Actor , making him only the second actor to win a posthumous Academy Award .
---
TAG: p
The Dark Knight appeared on several lists recognizing the best films of 2008, including those compiled by Ebert, The Hollywood Reporter , and the American Film Institute . [ as ] At the 13th Satellite Awards , The Dark Knight received one award for Sound Editing or Mixing ( Richard King , Lora Hirschberg , Gary Rizzo ). [ 250 ] A further four wins came at the 35th People's Choice Awards : Favorite Movie, Favorite Cast, Favorite Action Movie, and Favorite On-Screen Match-Up (Bale and Ledger), [ 2
---
TAG: p
Before The Dark Knight ' s release, film industry discourse focused on Ledger potentially earning an Academy Award nomination at the 81st Academy Awards in 2009, making him only the seventh person to be nominated posthumously, and if

In [34]:
accolades_heading = soup.find("h3", id="Accolades")

accolades_text_parts = []

parent = accolades_heading.parent

for element in parent.find_next_siblings():
    text = element.get_text(" ", strip=True)

    if text in ["Other releases", "Home media", "Merchandise and spin-offs"]:
        break

    if element.name in ["p", "div", "figure"]:
        accolades_text_parts.append(text)

accolades_text = " ".join(accolades_text_parts)

print(accolades_text[:2000])

Main article: List of accolades received by The Dark Knight Heath Ledger (2006) received the Academy Award for Best Supporting Actor , making him only the second actor to win a posthumous Academy Award . The Dark Knight appeared on several lists recognizing the best films of 2008, including those compiled by Ebert, The Hollywood Reporter , and the American Film Institute . [ as ] At the 13th Satellite Awards , The Dark Knight received one award for Sound Editing or Mixing ( Richard King , Lora Hirschberg , Gary Rizzo ). [ 250 ] A further four wins came at the 35th People's Choice Awards : Favorite Movie, Favorite Cast, Favorite Action Movie, and Favorite On-Screen Match-Up (Bale and Ledger), [ 251 ] as well as Best Action Movie and Best Supporting Actor (Ledger) at the 14th Critics' Choice Awards . [ 252 ] Howard and Zimmer were recognized for Best Motion Picture Score at the 51st Annual Grammy Awards . [ 253 ] Ledger won the film's only awards at the 15th Screen Actors Guild Awards , 

In [35]:
has_awards = True

In [36]:
dark_knight_result = {
    "Title": "The Dark Knight",
    "wikipedia_url": "https://en.wikipedia.org/wiki/The_Dark_Knight",
    "wikipedia_section_found": True,
    "has_awards": True,
    "awards_text": accolades_text
}

dark_knight_result

{'Title': 'The Dark Knight',
 'wikipedia_url': 'https://en.wikipedia.org/wiki/The_Dark_Knight',
 'wikipedia_section_found': True,
 'has_awards': True,
 'awards_text': 'Main article: List of accolades received by The Dark Knight Heath Ledger (2006) received the Academy Award for Best Supporting Actor , making him only the second actor to win a posthumous Academy Award . The Dark Knight appeared on several lists recognizing the best films of 2008, including those compiled by Ebert, The Hollywood Reporter , and the American Film Institute . [ as ] At the 13th Satellite Awards , The Dark Knight received one award for Sound Editing or Mixing ( Richard King , Lora Hirschberg , Gary Rizzo ). [ 250 ] A further four wins came at the 35th People\'s Choice Awards : Favorite Movie, Favorite Cast, Favorite Action Movie, and Favorite On-Screen Match-Up (Bale and Ledger), [ 251 ] as well as Best Action Movie and Best Supporting Actor (Ledger) at the 14th Critics\' Choice Awards . [ 252 ] Howard and Z

In [37]:
test_awards_df = pd.DataFrame([dark_knight_result])

test_awards_df

,Title,wikipedia_url,wikipedia_section_found,has_awards,awards_text
0,The Dark Knight,https://en.wikipedia.org/wiki/The_Dark_Knight,True,True,Main article: List of accolades received by Th...


In [38]:
test_awards_df.to_csv("wikipedia_awards_test.csv", index=False)

In [39]:
def scrape_awards(title):

    wiki_title = title.replace(" ", "_")
    url = f"https://en.wikipedia.org/wiki/{wiki_title}"

    try:
        response = requests.get(url, headers=headers, timeout=10)

        if response.status_code != 200:
            return False, None, None

        soup = BeautifulSoup(response.text, "html.parser")

        possible_award_ids = [
            "Accolades",
            "Awards",
            "Awards_and_nominations",
            "List_of_accolades"
        ]

        accolades_heading = None

        for award_id in possible_award_ids:
            accolades_heading = soup.find(id=award_id)

            if accolades_heading is not None:
                break

        if accolades_heading is None:
            return True, url, None

        accolades_text_parts = []

        parent = accolades_heading.parent

        for element in parent.find_next_siblings():

            text = element.get_text(" ", strip=True)

            if text in [
                "Other releases",
                "Home media",
                "Merchandise and spin-offs",
                "Themes and analysis",
                "Legacy"
            ]:
                break

            if element.name in ["p", "div", "figure"]:
                accolades_text_parts.append(text)

        accolades_text = " ".join(accolades_text_parts)

        return True, url, accolades_text

    except Exception:
        return False, None, None

In [40]:
page_found, wiki_url, awards_text = scrape_awards("Inception")

print(page_found)
print(wiki_url)

if awards_text:
    print(awards_text[:1000])
else:
    print("No awards text found")

True
https://en.wikipedia.org/wiki/Inception
No awards text found


In [41]:
url = "https://en.wikipedia.org/wiki/List_of_accolades_received_by_Inception"

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

print(response.status_code)
print(soup.title.text)

200
List of accolades received by Inception - Wikipedia


In [42]:
page_text = soup.get_text(" ", strip=True)

print(page_text[:1000])

List of accolades received by Inception - Wikipedia Jump to content Main menu Main menu move to sidebar hide Navigation Main page Contents Current events Random article About Wikipedia Contact us Contribute Help Learn to edit Community portal Recent changes Upload file Special pages Print/export Download as PDF Printable version In other projects Wikidata item Search Search Appearance Donate Create account Log in Personal tools Donate Create account Log in Contents move to sidebar hide (Top) 1 Accolades 2 Notes 3 References 4 External links Toggle the table of contents List of accolades received by Inception 2 languages 日本語 中文 Edit links Article Talk English Read Edit View history Tools Tools move to sidebar hide Actions Read Edit View history General What links here Related changes Upload file Permanent link Page information Cite this page Get shortened URL Print/export Download as PDF Printable version In other projects Wikidata item Appearance move to sidebar hide From Wikipedia, th

In [43]:
def scrape_awards(title):

    wiki_title = title.replace(" ", "_")

    urls_to_try = [
        f"https://en.wikipedia.org/wiki/List_of_accolades_received_by_{wiki_title}",
        f"https://en.wikipedia.org/wiki/{wiki_title}"
    ]

    for url in urls_to_try:

        try:
            response = requests.get(url, headers=headers, timeout=10)

            if response.status_code != 200:
                continue

            soup = BeautifulSoup(response.text, "html.parser")
            page_text = soup.get_text(" ", strip=True)

            award_keywords = [
                "Academy Award",
                "Golden Globe",
                "BAFTA",
                "award",
                "nomination",
                "won"
            ]

            has_awards = any(keyword.lower() in page_text.lower() for keyword in award_keywords)

            if has_awards:
                return True, url, page_text[:3000]

        except Exception:
            continue

    return False, None, None

In [44]:
page_found, wiki_url, awards_text = scrape_awards("Inception")

print(page_found)
print(wiki_url)
print(awards_text[:1000] if awards_text else "No awards text found")

True
https://en.wikipedia.org/wiki/List_of_accolades_received_by_Inception
List of accolades received by Inception - Wikipedia Jump to content Main menu Main menu move to sidebar hide Navigation Main page Contents Current events Random article About Wikipedia Contact us Contribute Help Learn to edit Community portal Recent changes Upload file Special pages Print/export Download as PDF Printable version In other projects Wikidata item Search Search Appearance Donate Create account Log in Personal tools Donate Create account Log in Contents move to sidebar hide (Top) 1 Accolades 2 Notes 3 References 4 External links Toggle the table of contents List of accolades received by Inception 2 languages 日本語 中文 Edit links Article Talk English Read Edit View history Tools Tools move to sidebar hide Actions Read Edit View history General What links here Related changes Upload file Permanent link Page information Cite this page Get shortened URL Print/export Download as PDF Printable version In othe

In [45]:
results = []

for _, row in sample_movies.iterrows():

    title = row["Title"]

    page_found, wiki_url, awards_text = scrape_awards(title)

    results.append({
        "Title": title,
        "Year": row["Year"],
        "IMDb Votes": row["IMDb Votes"],
        "Wikipedia Page Found": page_found,
        "Wikipedia URL": wiki_url,
        "Awards Text": awards_text,
        "has_awards": page_found
    })

    print(f"Done: {title}")

    time.sleep(1)

Done: The Dark Knight
Done: Inception
Done: Interstellar
Done: The Lord of the Rings: The Fellowship of the Ring
Done: The Lord of the Rings: The Return of the King
Done: The Avengers
Done: Avatar
Done: Avengers: Endgame
Done: Avengers: Infinity War
Done: Parasite
Done: Monsters, Inc.
Done: Spider-Man: No Way Home
Done: Spirited Away
Done: Toy Story 3
Done: Spider-Man
Done: Harry Potter and the Philosopher's Stone
Done: Shrek
Done: Spider-Man: Homecoming
Done: Harry Potter and the Prisoner of Azkaban
Done: Harry Potter and the Chamber of Secrets
Done: Harry Potter and the Goblet of Fire
Done: Harry Potter and the Order of the Phoenix
Done: Harry Potter and the Half-Blood Prince
Done: The Devil Wears Prada
Done: Real Steel
Done: Apocalypto
Done: Fifty Shades of Grey
Done: Scary Movie
Done: Toy Story 4
Done: Coraline
Done: Mortal Kombat
Done: Scary Movie 2
Done: Irreversible
Done: Nymphomaniac: Vol. II
Done: xXx: Return of Xander Cage
Done: The Human Centipede (First Sequence)
Done: Scar

In [46]:
awards_df = pd.DataFrame(results)

In [47]:
awards_df["has_awards"].value_counts()

has_awards
True     38
False    12
Name: count, dtype: int64

In [48]:
awards_df[awards_df["has_awards"] == False][
    ["Title", "Wikipedia URL"]
]

,Title,Wikipedia URL
5,The Avengers,NaN
23,The Devil Wears Prada,NaN
32,Irreversible,NaN
33,Nymphomaniac: Vol. II,NaN
37,Fifty Shades Freed,NaN
39,My Fault,NaN
43,Addicted,NaN
44,From Straight A's to XXX,NaN
45,Graphic Desires,NaN
46,Dime lo que quieres (de verdad),NaN


In [49]:
awards_df.to_csv("wikipedia_awards_scraped.csv", index=False)

In [50]:
final_df = df.merge(
    awards_df,
    on=["Title", "Year"],
    how="left"
)

In [51]:
final_df.shape

(815, 20)

In [52]:
final_df.head()

,TMDb_ID,IMDb_ID,Title,Year,Genre,Primary Genre,TMDb Rating,TMDb Votes,Popularity,Revenue,Budget,Runtime,IMDb Rating,IMDb Votes_x,Box Office,IMDb Votes_y,Wikipedia Page Found,Wikipedia URL,Awards Text,has_awards
0,1339713,tt37287335,Obsession,2026,"Horror, Thriller",Horror,7.917,841,581.7554,297445815.0,750000.0,108,8.2,50183.0,104757650.0,NaN,NaN,NaN,NaN,NaN
1,931285,tt17490712,Mortal Kombat II,2026,"Action, Fantasy, Adventure",Action,7.988,1449,519.4298,128857238.0,80000000.0,116,6.7,36627.0,77767462.0,NaN,NaN,NaN,NaN,NaN
2,936075,tt11378946,Michael,2026,"Music, Drama",Music,8.653,2635,503.0910,934009960.0,250000000.0,128,7.7,107813.0,340068767.0,NaN,NaN,NaN,NaN,NaN
3,1367220,tt33988385,Karuppu,2026,"Action, Fantasy",Action,7.059,17,384.3573,3345079.0,16000000.0,152,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,879945,tt16288638,The Unknown Man,2021,Drama,Drama,7.800,9,325.1904,NaN,NaN,23,7.0,53.0,NaN,53.0,True,https://en.wikipedia.org/wiki/The_Unknown_Man,The Unknown Man - Wikipedia Jump to content Ma...,True


Cleaning

In [53]:
final_df = final_df.rename(columns={"IMDb Votes_x": "IMDb Votes"})
final_df = final_df.drop(columns=["IMDb Votes_y"])

In [54]:
final_df.columns

Index(['TMDb_ID', 'IMDb_ID', 'Title', 'Year', 'Genre', 'Primary Genre',
       'TMDb Rating', 'TMDb Votes', 'Popularity', 'Revenue', 'Budget',
       'Runtime', 'IMDb Rating', 'IMDb Votes', 'Box Office',
       'Wikipedia Page Found', 'Wikipedia URL', 'Awards Text', 'has_awards'],
      dtype='str')

In [77]:
final_df = final_df.rename(
    columns={"IMDb Votes_x": "IMDb Votes"}
)

Some analysis 

In [55]:
final_df.groupby("has_awards")["IMDb Rating"].mean()

has_awards
False    5.900000
True     7.221053
Name: IMDb Rating, dtype: float64

In [56]:
final_df.groupby("has_awards")["IMDb Votes"].mean()

has_awards
False    227660.090909
True     849320.000000
Name: IMDb Votes, dtype: float64

In [57]:
final_df.columns.tolist()

['TMDb_ID',
 'IMDb_ID',
 'Title',
 'Year',
 'Genre',
 'Primary Genre',
 'TMDb Rating',
 'TMDb Votes',
 'Popularity',
 'Revenue',
 'Budget',
 'Runtime',
 'IMDb Rating',
 'IMDb Votes',
 'Box Office',
 'Wikipedia Page Found',
 'Wikipedia URL',
 'Awards Text',
 'has_awards']

In [58]:
final_df[["Title", "IMDb Rating", "has_awards"]].head()

,Title,IMDb Rating,has_awards
0,Obsession,8.2,NaN
1,Mortal Kombat II,6.7,NaN
2,Michael,7.7,NaN
3,Karuppu,NaN,NaN
4,The Unknown Man,7.0,True


In [59]:
final_df.groupby("has_awards")["IMDb Rating"].mean()

has_awards
False    5.900000
True     7.221053
Name: IMDb Rating, dtype: float64

In [60]:
#Distribution of Awards Information
final_df["has_awards"].value_counts()

has_awards
True     38
False    12
Name: count, dtype: int64

In [61]:
#Average IMDb Rating by Awards Status

final_df.groupby("has_awards")["IMDb Rating"].mean()

has_awards
False    5.900000
True     7.221053
Name: IMDb Rating, dtype: float64

In [62]:
final_df.shape

(815, 19)

In [63]:
final_df[[
    "Title",
    "IMDb Rating",
    "has_awards"
]].sample(10)

,Title,IMDb Rating,has_awards
472,Kraven the Hunter,5.5,NaN
720,RDX: Robert Dony Xavier,7.0,NaN
556,The Torture Club,4.4,NaN
657,A Good Day to Die Hard,5.2,NaN
810,Percy Jackson & the Olympians: The Lightning T...,NaN,NaN
377,Sex Trip,NaN,NaN
53,Scary Movie,6.3,True
268,Exit 8,6.4,NaN
590,Biker,NaN,NaN
446,"My Teacher, My Obsession",4.4,NaN


In [64]:
final_df["has_awards"].value_counts(dropna=False)

has_awards
NaN      765
True      38
False     12
Name: count, dtype: int64

In [65]:
final_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 815 entries, 0 to 814
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   TMDb_ID               815 non-null    int64  
 1   IMDb_ID               815 non-null    str    
 2   Title                 815 non-null    str    
 3   Year                  815 non-null    int64  
 4   Genre                 815 non-null    str    
 5   Primary Genre         815 non-null    str    
 6   TMDb Rating           815 non-null    float64
 7   TMDb Votes            815 non-null    int64  
 8   Popularity            815 non-null    float64
 9   Revenue               553 non-null    float64
 10  Budget                547 non-null    float64
 11  Runtime               815 non-null    int64  
 12  IMDb Rating           687 non-null    float64
 13  IMDb Votes            702 non-null    float64
 14  Box Office            498 non-null    float64
 15  Wikipedia Page Found  50 non-null 

In [66]:
final_df.to_csv("movie_data_with_scraping.csv", index=False)